# Benchmark Dashboard

Kompakte Analyseoberflaeche fuer die Benchmark-Runs aus den `run.json`-Artefakten. Das Notebook liest die JSON-Daten direkt ein, bereitet sie mit `pandas` auf und erzeugt interaktive, notebook-taugliche Charts mit `plotly`.

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "plotly": "plotly",
    "numpy": "numpy",
    "ipython": "IPython",
    "nbformat": "nbformat",
}
missing_packages = [package for package, module_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]

if missing_packages:
    print("Installiere fehlende Pakete:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Alle benoetigten Pakete sind bereits installiert.")


Alle benoetigten Pakete sind bereits installiert.


In [2]:
import html
import json
from pathlib import Path
from typing import Iterable, Optional, Union

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots

# Notebook-Renderer mit Fallback fuer Umgebungen, die nur statische HTML-Ausgabe sauber unterstuetzen.
pio.renderers.default = "notebook_connected" if "notebook_connected" in pio.renderers else "iframe"

def resolve_runs_source(default_relative_path: str = ".benchmarks/runs") -> Path:
    """Sucht den Runs-Ordner robust relativ zum aktuellen Arbeitsverzeichnis."""
    search_roots = [Path.cwd(), *Path.cwd().parents]
    
    for root in search_roots:
        candidate = root / default_relative_path
        if candidate.exists():
            return candidate.resolve()

    for root in search_roots:
        if (root / "benchmark.yaml").exists():
            candidate = root / default_relative_path
            if candidate.exists():
                return candidate.resolve()

    raise FileNotFoundError(
        f"Konnte den Runs-Ordner '{default_relative_path}' nicht finden. "
        f"Aktuelles Arbeitsverzeichnis: {Path.cwd()}"
    )


RUNS_SOURCE = resolve_runs_source()
TIME_DECIMALS = 1
PREFERRED_TOOL_ORDER = ["codex", "opencode", "claude"]
TOOL_COLORS = {
    "codex": "#1f77b4",
    "opencode": "#111111",
    "claude": "#ff7f0e",
}
FALLBACK_COLORS = ["#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]


In [3]:
def seconds_label(value: Optional[Union[float, int]], decimals: int = TIME_DECIMALS) -> str:
    """Formatiert Laufzeiten einheitlich in Sekunden."""
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.{decimals}f} s".replace(",", "_").replace(".", ",").replace("_", ".")


def mean_seconds(series: pd.Series) -> float:
    """Robuste Mittelwert-Helferfunktion fuer Aggregationen."""
    return float(series.mean()) if len(series) else float("nan")


def median_seconds(series: pd.Series) -> float:
    """Robuste Median-Helferfunktion fuer Aggregationen."""
    return float(series.median()) if len(series) else float("nan")


def tool_model_label(tool: str, model: str) -> str:
    """Kompakte Beschriftung fuer Tool-Model-Kombinationen."""
    return f"{tool} | {model}"


def ordered_tools(tools: Iterable[str]) -> list[str]:
    tools = list(dict.fromkeys(tools))
    preferred = [tool for tool in PREFERRED_TOOL_ORDER if tool in tools]
    remaining = sorted(tool for tool in tools if tool not in preferred)
    return preferred + remaining


def build_tool_palette(tools: Iterable[str]) -> dict[str, str]:
    tool_names = ordered_tools(tools)
    palette = {tool: TOOL_COLORS[tool] for tool in tool_names if tool in TOOL_COLORS}
    fallback_index = 0
    for tool in tool_names:
        if tool not in palette:
            palette[tool] = FALLBACK_COLORS[fallback_index % len(FALLBACK_COLORS)]
            fallback_index += 1
    return palette


def dynamic_chart_height(items_per_task: pd.Series, min_height: int = 360, max_height: int = 1800) -> int:
    """Passt die Hoehe an die Anzahl der Labels und Subplots an."""
    total_items = int(items_per_task.sum())
    task_count = max(int(len(items_per_task)), 1)
    height = 120 + task_count * 110 + total_items * 34
    return max(min_height, min(max_height, height))


def load_run_records(source: Union[str, Path]) -> list[dict]:
    """Laedt entweder eine einzelne JSON-Datei oder rekursiv alle run.json-Dateien aus einem Ordner."""
    source_path = Path(source).expanduser()

    if source_path.is_dir():
        files = sorted(source_path.rglob("run.json"))
        if not files:
            raise FileNotFoundError(f"Keine run.json-Dateien unter {source_path} gefunden.")
        return [{**json.loads(file.read_text(encoding="utf-8")), "SourceFile": str(file)} for file in files]

    if source_path.is_file():
        payload = json.loads(source_path.read_text(encoding="utf-8"))
        if isinstance(payload, list):
            return payload
        if isinstance(payload, dict):
            return [payload]
        raise ValueError(f"Unerwartetes JSON-Format in {source_path}.")

    raise FileNotFoundError(f"Quelle nicht gefunden: {source_path}")


def prepare_runs_dataframe(records: list[dict]) -> pd.DataFrame:
    """Normalisiert die JSON-Records in ein DataFrame fuer die weitere Analyse."""
    rename_map = {
        "RunId": "run_id",
        "Tool": "tool",
        "Model": "model",
        "Task": "task",
        "RepoPath": "repo_path",
        "Status": "status",
        "ExitCode": "exit_code",
        "TimedOut": "timed_out",
        "ArtifactDirectory": "artifact_directory",
        "StartedAtUtc": "started_at_utc",
        "FinishedAtUtc": "finished_at_utc",
        "DurationSeconds": "duration_seconds",
        "QualityRating": "quality_rating",
        "FailureReason": "failure_reason",
        "SourceFile": "source_file",
    }

    runs = pd.DataFrame(records).rename(columns=rename_map)
    if runs.empty:
        raise ValueError("Es wurden keine Benchmark-Daten geladen.")

    runs["duration_seconds"] = pd.to_numeric(runs["duration_seconds"], errors="coerce")
    runs["started_at_utc"] = pd.to_datetime(runs["started_at_utc"], utc=True, errors="coerce")
    runs["finished_at_utc"] = pd.to_datetime(runs["finished_at_utc"], utc=True, errors="coerce")
    runs["timed_out"] = runs["timed_out"].fillna(False).astype(bool)
    runs["tool_model"] = runs.apply(lambda row: tool_model_label(row["tool"], row["model"]), axis=1)
    runs["duration_label"] = runs["duration_seconds"].map(seconds_label)
    runs["started_label"] = runs["started_at_utc"].dt.strftime("%Y-%m-%d %H:%M:%S UTC")
    runs["finished_label"] = runs["finished_at_utc"].dt.strftime("%Y-%m-%d %H:%M:%S UTC")

    return runs.sort_values(["task", "tool", "model", "started_at_utc"], kind="stable").reset_index(drop=True)


def aggregate_mean_duration(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    """Erstellt eine wiederverwendbare Aggregation fuer mittlere Laufzeiten."""
    return (
        df.groupby(group_columns, as_index=False)
        .agg(
            avg_duration_seconds=("duration_seconds", mean_seconds),
            median_duration_seconds=("duration_seconds", median_seconds),
            run_count=("run_id", "count"),
        )
    )


def add_tool_legend(fig: go.Figure, palette: dict[str, str]) -> None:
    """Fuegt eine saubere, einmalige Legende fuer die Tool-Farben hinzu."""
    for tool, color in palette.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(size=10, color=color),
                name=tool,
                legendgroup=tool,
                showlegend=True,
                hoverinfo="skip",
            )
        )


def apply_base_layout(fig: go.Figure, title: str, height: int, x_title: str) -> None:
    """Einheitliches Styling fuer alle Diagramme."""
    fig.update_layout(
        title=dict(text=title, x=0.0, xanchor="left"),
        height=height,
        template="plotly_white",
        hoverlabel=dict(bgcolor="white"),
        margin=dict(l=24, r=40, t=72, b=36),
        font=dict(family="Aptos, Segoe UI, sans-serif", size=12),
        legend=dict(orientation="h", x=1.0, xanchor="right", y=1.06, yanchor="bottom"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        bargap=0.24,
    )
    fig.update_xaxes(title_text=x_title, gridcolor="#dfe6ee", zeroline=False)
    fig.update_yaxes(title_text=None, automargin=True)


In [4]:
records = load_run_records(RUNS_SOURCE)
runs = prepare_runs_dataframe(records)
tool_palette = build_tool_palette(runs["tool"])

summary_html = f"""
<div style='margin: 4px 0 18px; color: #445468; font-family: Aptos, Segoe UI, sans-serif;'>
  Quelle: <code>{html.escape(str(RUNS_SOURCE))}</code><br>
  Geladene Runs: <strong>{len(runs)}</strong> | Tasks: <strong>{runs['task'].nunique()}</strong> | Tools: <strong>{runs['tool'].nunique()}</strong> | Modelle: <strong>{runs['model'].nunique()}</strong>
</div>
"""
display(HTML(summary_html))
runs[["run_id", "task", "tool", "model", "duration_seconds", "status"]].head()


,run_id,task,tool,model,duration_seconds,status
0,20260324153820-d893832c,run-build-checks,claude,glm-4.7-flash:q4_K_M,278.368,Succeeded
1,20260324153820-bc2c1331,run-build-checks,claude,gpt-oss:120b,94.256,Succeeded
2,20260324153820-8c763f18,run-build-checks,claude,gpt-oss:20b,309.382,Succeeded
3,20260324142821-0ca56b82,run-build-checks,claude,qwen3-coder-next:q4_K_M,100.338,Succeeded
4,20260324142821-ec791166,run-build-checks,claude,qwen3-coder-next:q8_0,96.185,Succeeded


## Uebersicht

In [5]:
def display_kpi_cards(df: pd.DataFrame) -> None:
    fastest_run = df.nsmallest(1, "duration_seconds").iloc[0]
    median_duration = median_seconds(df["duration_seconds"])

    cards = [
        {
            "title": "Alle Runs",
            "value": f"{len(df):,}".replace(",", "."),
            "subtitle": f"{df['task'].nunique()} Tasks | {df['tool'].nunique()} Tools | {df['model'].nunique()} Modelle",
        },
        {
            "title": "Schnellster Run",
            "value": seconds_label(fastest_run['duration_seconds']),
            "subtitle": f"{fastest_run['tool']} | {fastest_run['task']}",
        },
        {
            "title": "Median Laufzeit",
            "value": seconds_label(median_duration),
            "subtitle": "Ueber alle Runs",
        },
    ]

    card_html = """
    <div style='display:grid; grid-template-columns:repeat(3, minmax(220px, 1fr)); gap:16px; margin:6px 0 22px;'>
    """
    for card in cards:
        card_html += f"""
        <div style='border:1px solid #d9e1ea; border-radius:16px; padding:18px 20px; background:linear-gradient(180deg, #ffffff 0%, #f7fafc 100%); box-shadow:0 6px 18px rgba(16, 24, 40, 0.05);'>
          <div style='font:600 12px Aptos, Segoe UI, sans-serif; letter-spacing:0.04em; text-transform:uppercase; color:#5f7288; margin-bottom:10px;'>
            {html.escape(card['title'])}
          </div>
          <div style='font:700 28px Aptos, Segoe UI, sans-serif; color:#16202a; margin-bottom:8px;'>
            {html.escape(card['value'])}
          </div>
          <div style='font:500 13px Aptos, Segoe UI, sans-serif; color:#4d6278;'>
            {html.escape(card['subtitle'])}
          </div>
        </div>
        """
    card_html += "</div>"

    display(HTML(card_html))


display_kpi_cards(runs)


## Durchschnittliche Dauer nach Tool und Task

In [6]:
def plot_average_duration_by_tool_task(df: pd.DataFrame, palette: dict[str, str]) -> go.Figure:
    chart_df = aggregate_mean_duration(df, ["task", "tool"]).sort_values(["task", "avg_duration_seconds", "tool"], kind="stable")
    tasks = chart_df["task"].drop_duplicates().tolist()
    items_per_task = chart_df.groupby("task")["tool"].nunique()

    fig = make_subplots(
        rows=len(tasks),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=[f"Task: {task}" for task in tasks],
    )

    for row_index, task in enumerate(tasks, start=1):
        task_df = chart_df.loc[chart_df["task"] == task].sort_values("avg_duration_seconds", kind="stable")
        fig.add_trace(
            go.Bar(
                x=task_df["avg_duration_seconds"],
                y=task_df["tool"],
                orientation="h",
                marker=dict(color=[palette[tool] for tool in task_df["tool"]]),
                text=[seconds_label(value) for value in task_df["avg_duration_seconds"]],
                textposition="outside",
                cliponaxis=False,
                customdata=task_df[["task", "tool", "run_count"]].to_numpy(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Tool: %{customdata[1]}<br>"
                    "Durchschnitt: %{x:.1f} s<br>"
                    "Runs: %{customdata[2]}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row_index,
            col=1,
        )
        fig.update_yaxes(autorange="reversed", row=row_index, col=1)

    add_tool_legend(fig, palette)
    apply_base_layout(
        fig,
        title="Durchschnittliche Dauer nach Tool und Task",
        height=dynamic_chart_height(items_per_task),
        x_title="Durchschnittliche Dauer (Sekunden)",
    )
    return fig


plot_average_duration_by_tool_task(runs, tool_palette).show()


## Schnellste Modell-Konfigurationen im Mittel

In [7]:
def plot_fastest_model_configs(df: pd.DataFrame, palette: dict[str, str]) -> go.Figure:
    chart_df = (
        aggregate_mean_duration(df, ["task", "tool", "model"])
        .assign(tool_model=lambda data: data.apply(lambda row: tool_model_label(row["tool"], row["model"]), axis=1))
        .sort_values(["task", "avg_duration_seconds", "tool", "model"], kind="stable")
    )
    tasks = chart_df["task"].drop_duplicates().tolist()
    items_per_task = chart_df.groupby("task")["tool_model"].nunique()

    fig = make_subplots(
        rows=len(tasks),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=[f"Task: {task}" for task in tasks],
    )

    for row_index, task in enumerate(tasks, start=1):
        task_df = chart_df.loc[chart_df["task"] == task].sort_values("avg_duration_seconds", kind="stable")
        fig.add_trace(
            go.Bar(
                x=task_df["avg_duration_seconds"],
                y=task_df["tool_model"],
                orientation="h",
                marker=dict(color=[palette[tool] for tool in task_df["tool"]]),
                text=[seconds_label(value) for value in task_df["avg_duration_seconds"]],
                textposition="outside",
                cliponaxis=False,
                customdata=task_df[["task", "tool", "model", "run_count"]].to_numpy(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Tool: %{customdata[1]}<br>"
                    "Model: %{customdata[2]}<br>"
                    "Durchschnitt: %{x:.1f} s<br>"
                    "Runs: %{customdata[3]}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row_index,
            col=1,
        )
        fig.update_yaxes(autorange="reversed", row=row_index, col=1)

    add_tool_legend(fig, palette)
    apply_base_layout(
        fig,
        title="Schnellste Modell-Konfigurationen im Mittel",
        height=dynamic_chart_height(items_per_task, min_height=420),
        x_title="Durchschnittliche Dauer (Sekunden)",
    )
    return fig


plot_fastest_model_configs(runs, tool_palette).show()


## Einzelne Runs

In [8]:
def plot_individual_runs(df: pd.DataFrame, palette: dict[str, str]) -> go.Figure:
    plot_df = df.sort_values(["task", "duration_seconds", "tool", "model", "started_at_utc"], kind="stable").copy()
    tasks = plot_df["task"].drop_duplicates().tolist()
    items_per_task = plot_df.groupby("task")["tool_model"].nunique()

    fig = make_subplots(
        rows=len(tasks),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=[f"Task: {task}" for task in tasks],
    )

    for row_index, task in enumerate(tasks, start=1):
        task_df = plot_df.loc[plot_df["task"] == task].copy()
        category_order = (
            task_df.groupby("tool_model")["duration_seconds"]
            .mean()
            .sort_values(kind="stable")
            .index.tolist()
        )

        for tool in ordered_tools(task_df["tool"]):
            tool_df = task_df.loc[task_df["tool"] == tool]
            if tool_df.empty:
                continue

            customdata = tool_df[[
                "run_id",
                "tool",
                "model",
                "task",
                "status",
                "duration_label",
                "started_label",
                "finished_label",
            ]].to_numpy()

            fig.add_trace(
                go.Scatter(
                    x=tool_df["duration_seconds"],
                    y=tool_df["tool_model"],
                    mode="markers",
                    marker=dict(
                        color=palette[tool],
                        size=11,
                        line=dict(color="white", width=0.9),
                        opacity=0.92,
                    ),
                    name=tool,
                    legendgroup=tool,
                    showlegend=(row_index == 1),
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{customdata[3]}</b><br>"
                        "Run: %{customdata[0]}<br>"
                        "Tool: %{customdata[1]}<br>"
                        "Model: %{customdata[2]}<br>"
                        "Status: %{customdata[4]}<br>"
                        "Dauer: %{customdata[5]}<br>"
                        "Start: %{customdata[6]}<br>"
                        "Ende: %{customdata[7]}<extra></extra>"
                    ),
                ),
                row=row_index,
                col=1,
            )

        fig.update_yaxes(categoryorder="array", categoryarray=category_order[::-1], autorange="reversed", row=row_index, col=1)

    apply_base_layout(
        fig,
        title="Einzelne Runs",
        height=dynamic_chart_height(items_per_task, min_height=420),
        x_title="Dauer (Sekunden)",
    )
    fig.update_traces(marker_symbol="circle")
    return fig


plot_individual_runs(runs, tool_palette).show()
